# Graph Algorithms - Worked Examples

This notebook provides 12 practical implementations of classic graph algorithms with applications to machine learning problems. Graph algorithms are fundamental for understanding network structures, optimizing paths, and powering graph neural networks.

**Topics covered:**
1. Breadth-First Search (BFS)
2. Depth-First Search (DFS)
3. Dijkstra's Shortest Path Algorithm
4. Bellman-Ford Algorithm
5. Floyd-Warshall All-Pairs Shortest Path
6. Prim's MST Algorithm
7. Kruskal's MST Algorithm
8. Topological Sort
9. Strongly Connected Components (Kosaraju/Tarjan)
10. Graph Coloring
11. Maximum Flow (Ford-Fulkerson)
12. PageRank

In [ ]:
# Common imports for all examples
import numpy as np
import networkx as nx
from collections import defaultdict, deque
import heapq
from typing import List, Tuple, Dict, Set, Optional
import matplotlib.pyplot as plt

## Example 1: Breadth-First Search (BFS)

BFS explores a graph level-by-level, visiting all vertices at distance $k$ before visiting vertices at distance $k+1$.

**Properties:**
- Finds shortest path in unweighted graphs
- Time complexity: $O(V + E)$
- Space complexity: $O(V)$

**Applications in ML:**
- K-hop neighborhood aggregation in GNNs
- Feature propagation in semi-supervised learning
- Computing graph distances for similarity metrics

In [ ]:
def bfs(adj_list: Dict[int, List[int]], start: int) -> Tuple[Dict, Dict]:
    """
    BFS traversal returning distances and parents.
    
    Args:
        adj_list: Adjacency list representation of graph
        start: Starting vertex
        
    Returns:
        distances: Shortest distance from start to each vertex
        parents: Parent in BFS tree (for path reconstruction)
    """
    distances = {start: 0}
    parents = {start: None}
    queue = deque([start])
    
    while queue:
        u = queue.popleft()
        for v in adj_list.get(u, []):
            if v not in distances:
                distances[v] = distances[u] + 1
                parents[v] = u
                queue.append(v)
    
    return distances, parents


def reconstruct_path(parents: Dict, start: int, end: int) -> List[int]:
    """Reconstruct shortest path from start to end."""
    if end not in parents:
        return []
    
    path = []
    current = end
    while current is not None:
        path.append(current)
        current = parents[current]
    return path[::-1]


def k_hop_neighbors(adj_list: Dict[int, List[int]], start: int, k: int) -> List[int]:
    """Find all vertices within k hops of start (useful for GNN receptive fields)."""
    distances, _ = bfs(adj_list, start)
    return [v for v, d in distances.items() if d <= k and v != start]


# Create example graph
adj_list = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4]
}

# Visualize with NetworkX
G = nx.Graph(adj_list)
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_color='lightblue', 
        node_size=500, font_size=16, font_weight='bold')
plt.title("Example Graph for BFS")
plt.show()

# Run BFS
distances, parents = bfs(adj_list, 0)
print(f"BFS from vertex 0:")
print(f"  Distances: {distances}")
print(f"  Shortest path 0 → 5: {reconstruct_path(parents, 0, 5)}")
print(f"  2-hop neighbors of 0: {k_hop_neighbors(adj_list, 0, 2)}")

## Example 2: Depth-First Search (DFS)

DFS explores as deep as possible along each branch before backtracking. It uses a stack (or recursion) to track the exploration path.

**Properties:**
- Time complexity: $O(V + E)$
- Space complexity: $O(V)$ for the recursion stack

**Applications:**
- Topological sorting
- Cycle detection
- Finding connected components
- Backtracking algorithms

In [ ]:
def dfs_recursive(adj_list: Dict[int, List[int]], start: int) -> List[int]:
    """Simple DFS returning visited order (recursive)."""
    visited = set()
    order = []
    
    def dfs(u):
        visited.add(u)
        order.append(u)
        for v in adj_list.get(u, []):
            if v not in visited:
                dfs(v)
    
    dfs(start)
    return order


def dfs_iterative(adj_list: Dict[int, List[int]], start: int) -> List[int]:
    """Iterative DFS using explicit stack."""
    visited = set()
    order = []
    stack = [start]
    
    while stack:
        u = stack.pop()
        if u not in visited:
            visited.add(u)
            order.append(u)
            # Add neighbors in reverse for consistent ordering
            for v in reversed(adj_list.get(u, [])):
                if v not in visited:
                    stack.append(v)
    
    return order


def dfs_with_timestamps(adj_list: Dict[int, List[int]]) -> Tuple[Dict, Dict]:
    """
    DFS with discovery and finish times.
    Useful for topological sort and SCC detection.
    """
    discovery = {}
    finish = {}
    time = [0]
    
    def dfs(u):
        time[0] += 1
        discovery[u] = time[0]
        
        for v in adj_list.get(u, []):
            if v not in discovery:
                dfs(v)
        
        time[0] += 1
        finish[u] = time[0]
    
    for v in adj_list:
        if v not in discovery:
            dfs(v)
    
    return discovery, finish


# Directed graph example
adj_list_directed = {
    0: [1, 2],
    1: [3],
    2: [3, 4],
    3: [5],
    4: [5],
    5: []
}

# Visualize
G_dir = nx.DiGraph(adj_list_directed)
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G_dir, seed=42)
nx.draw(G_dir, pos, with_labels=True, node_color='lightgreen', 
        node_size=500, font_size=16, font_weight='bold', arrows=True)
plt.title("Directed Graph for DFS")
plt.show()

# Run DFS
print(f"DFS order (recursive): {dfs_recursive(adj_list_directed, 0)}")
print(f"DFS order (iterative): {dfs_iterative(adj_list_directed, 0)}")

discovery, finish = dfs_with_timestamps(adj_list_directed)
print(f"\nDFS timestamps:")
for v in sorted(discovery.keys()):
    print(f"  Vertex {v}: discover={discovery[v]}, finish={finish[v]}")

## Example 3: Dijkstra's Shortest Path Algorithm

Dijkstra's algorithm finds the shortest path from a source vertex to all other vertices in a weighted graph with **non-negative** edge weights.

**Algorithm:**
1. Initialize distances: source = 0, all others = ∞
2. Use a priority queue ordered by distance
3. For each vertex, relax edges to neighbors

**Time complexity:** $O((V + E) \log V)$ with binary heap

**Applications:** Route planning, network routing, similarity search in metric spaces

In [ ]:
def dijkstra(adj_list: Dict[int, List[Tuple[int, float]]], 
             start: int) -> Tuple[Dict, Dict]:
    """
    Dijkstra's algorithm using min-heap.
    
    Args:
        adj_list: {vertex: [(neighbor, weight), ...]}
        start: Source vertex
        
    Returns:
        distances: Shortest distance to each vertex
        parents: Parent in shortest path tree
    """
    distances = {start: 0}
    parents = {start: None}
    pq = [(0, start)]  # (distance, vertex)
    visited = set()
    
    while pq:
        d, u = heapq.heappop(pq)
        
        if u in visited:
            continue
        visited.add(u)
        
        for v, weight in adj_list.get(u, []):
            new_dist = d + weight
            if v not in distances or new_dist < distances[v]:
                distances[v] = new_dist
                parents[v] = u
                heapq.heappush(pq, (new_dist, v))
    
    return distances, parents


# Create weighted graph
weighted_adj = {
    0: [(1, 4), (2, 1)],
    1: [(3, 1)],
    2: [(1, 2), (3, 5)],
    3: [(4, 3)],
    4: []
}

# Visualize with NetworkX
G_weighted = nx.DiGraph()
for u, edges in weighted_adj.items():
    for v, w in edges:
        G_weighted.add_edge(u, v, weight=w)

plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G_weighted, seed=42)
nx.draw(G_weighted, pos, with_labels=True, node_color='lightyellow', 
        node_size=500, font_size=16, font_weight='bold', arrows=True)
edge_labels = nx.get_edge_attributes(G_weighted, 'weight')
nx.draw_networkx_edge_labels(G_weighted, pos, edge_labels=edge_labels)
plt.title("Weighted Graph for Dijkstra's Algorithm")
plt.show()

# Run Dijkstra
distances, parents = dijkstra(weighted_adj, 0)
print(f"Shortest distances from vertex 0:")
for v in sorted(distances.keys()):
    print(f"  To {v}: {distances[v]}")

path_to_4 = reconstruct_path(parents, 0, 4)
print(f"\nShortest path 0 → 4: {path_to_4} (cost: {distances[4]})")

## Example 4: Bellman-Ford Algorithm

Bellman-Ford handles graphs with **negative edge weights** and can detect **negative cycles**.

**Algorithm:**
1. Initialize distances: source = 0, all others = ∞
2. Relax all edges $|V| - 1$ times
3. Check for negative cycles by attempting one more relaxation

**Time complexity:** $O(VE)$

**Applications:** Arbitrage detection in currency exchange, network analysis with losses

In [ ]:
def bellman_ford(vertices: List[int], 
                 edges: List[Tuple[int, int, float]],
                 start: int) -> Tuple[Dict, Dict, bool]:
    """
    Bellman-Ford algorithm.
    
    Args:
        vertices: List of vertex IDs
        edges: List of (source, target, weight)
        start: Source vertex
        
    Returns:
        distances, parents, has_negative_cycle
    """
    distances = {v: float('inf') for v in vertices}
    parents = {v: None for v in vertices}
    distances[start] = 0
    
    # Relax all edges |V| - 1 times
    for _ in range(len(vertices) - 1):
        for u, v, w in edges:
            if distances[u] != float('inf') and distances[u] + w < distances[v]:
                distances[v] = distances[u] + w
                parents[v] = u
    
    # Check for negative cycles
    has_negative_cycle = False
    for u, v, w in edges:
        if distances[u] != float('inf') and distances[u] + w < distances[v]:
            has_negative_cycle = True
            break
    
    return distances, parents, has_negative_cycle


# Graph with negative weights (but no negative cycle)
vertices = [0, 1, 2, 3, 4]
edges = [
    (0, 1, 4), (0, 2, 1),
    (1, 3, 1),
    (2, 1, -2), (2, 3, 5),  # Negative weight edge!
    (3, 4, 3)
]

print("Graph with negative weights:")
for u, v, w in edges:
    print(f"  {u} --({w})--> {v}")

distances, parents, has_neg_cycle = bellman_ford(vertices, edges, 0)

print(f"\nShortest distances from vertex 0:")
for v in sorted(distances.keys()):
    print(f"  To {v}: {distances[v]}")
print(f"Has negative cycle: {has_neg_cycle}")

# Test with negative cycle
print("\n--- Adding negative cycle ---")
edges_neg_cycle = edges + [(4, 2, -8)]  # Creates negative cycle
_, _, has_neg_cycle = bellman_ford(vertices, edges_neg_cycle, 0)
print(f"Has negative cycle: {has_neg_cycle}")

## Example 5: Floyd-Warshall All-Pairs Shortest Path

Floyd-Warshall computes shortest paths between **all pairs** of vertices simultaneously.

**Algorithm:** Dynamic programming approach where $dist[i][j]^k$ = shortest path from $i$ to $j$ using only vertices $\{0, ..., k\}$ as intermediates.

$$dist[i][j]^k = \min(dist[i][j]^{k-1}, dist[i][k]^{k-1} + dist[k][j]^{k-1})$$

**Time complexity:** $O(V^3)$
**Space complexity:** $O(V^2)$

**Applications:** Computing all pairwise similarities, graph kernel computations

In [ ]:
def floyd_warshall(adj_matrix: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Floyd-Warshall algorithm.
    
    Args:
        adj_matrix: Adjacency matrix (inf for no edge)
        
    Returns:
        dist: Distance matrix
        next_hop: Next vertex matrix for path reconstruction
    """
    n = len(adj_matrix)
    dist = adj_matrix.copy()
    next_hop = np.zeros((n, n), dtype=int)
    
    # Initialize next_hop
    for i in range(n):
        for j in range(n):
            if i != j and dist[i, j] < np.inf:
                next_hop[i, j] = j
            else:
                next_hop[i, j] = -1
    
    # Main algorithm
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i, k] + dist[k, j] < dist[i, j]:
                    dist[i, j] = dist[i, k] + dist[k, j]
                    next_hop[i, j] = next_hop[i, k]
    
    return dist, next_hop


def reconstruct_fw_path(next_hop: np.ndarray, i: int, j: int) -> List[int]:
    """Reconstruct shortest path from i to j."""
    if next_hop[i, j] == -1:
        return []
    
    path = [i]
    while i != j:
        i = next_hop[i, j]
        path.append(i)
    return path


# Create adjacency matrix
INF = np.inf
adj = np.array([
    [0, 3, INF, 7],
    [8, 0, 2, INF],
    [5, INF, 0, 1],
    [2, INF, INF, 0]
])

print("Adjacency matrix (∞ = no edge):")
print(np.where(adj == INF, '∞', adj.astype(int)))

dist, next_hop = floyd_warshall(adj)

print("\nAll-pairs shortest distances:")
print(dist.astype(int))

print("\nShortest paths:")
for i in range(4):
    for j in range(4):
        if i != j:
            path = reconstruct_fw_path(next_hop, i, j)
            if path:
                print(f"  {i} → {j}: {path} (cost: {int(dist[i, j])})")

## Example 6: Prim's MST Algorithm

Prim's algorithm constructs a **Minimum Spanning Tree (MST)** by growing the tree from a starting vertex, always adding the minimum weight edge that connects a tree vertex to a non-tree vertex.

**Time complexity:** $O(E \log V)$ with binary heap

**Applications:** Network design, clustering, approximation algorithms

In [ ]:
def prim(n: int, adj_list: Dict[int, List[Tuple[int, float]]]) -> List[Tuple[int, int, float]]:
    """
    Prim's algorithm using min-heap.
    
    Args:
        n: Number of vertices
        adj_list: {vertex: [(neighbor, weight), ...]}
        
    Returns:
        List of edges in MST: [(u, v, weight), ...]
    """
    mst = []
    visited = set([0])
    
    # Add all edges from vertex 0
    edges = [(w, 0, v) for v, w in adj_list.get(0, [])]
    heapq.heapify(edges)
    
    while edges and len(mst) < n - 1:
        w, u, v = heapq.heappop(edges)
        
        if v in visited:
            continue
        
        visited.add(v)
        mst.append((u, v, w))
        
        # Add edges from new vertex
        for neighbor, weight in adj_list.get(v, []):
            if neighbor not in visited:
                heapq.heappush(edges, (weight, v, neighbor))
    
    return mst


# Create undirected weighted graph
edges_list = [
    (0, 1, 4), (0, 2, 3),
    (1, 2, 1), (1, 3, 2),
    (2, 3, 4), (2, 4, 3),
    (3, 4, 2), (3, 5, 1),
    (4, 5, 6)
]

adj_list_undirected = defaultdict(list)
for u, v, w in edges_list:
    adj_list_undirected[u].append((v, w))
    adj_list_undirected[v].append((u, w))

# Visualize original graph
G_mst = nx.Graph()
for u, v, w in edges_list:
    G_mst.add_edge(u, v, weight=w)

plt.figure(figsize=(10, 5))
pos = nx.spring_layout(G_mst, seed=42)

plt.subplot(1, 2, 1)
nx.draw(G_mst, pos, with_labels=True, node_color='lightcoral', 
        node_size=500, font_size=14, font_weight='bold')
edge_labels = nx.get_edge_attributes(G_mst, 'weight')
nx.draw_networkx_edge_labels(G_mst, pos, edge_labels=edge_labels)
plt.title("Original Graph")

# Run Prim's
mst = prim(6, adj_list_undirected)
total_weight = sum(w for _, _, w in mst)

# Visualize MST
G_mst_result = nx.Graph()
for u, v, w in mst:
    G_mst_result.add_edge(u, v, weight=w)

plt.subplot(1, 2, 2)
nx.draw(G_mst_result, pos, with_labels=True, node_color='lightgreen', 
        node_size=500, font_size=14, font_weight='bold', edge_color='green', width=2)
mst_labels = {(u, v): w for u, v, w in mst}
nx.draw_networkx_edge_labels(G_mst_result, pos, edge_labels=mst_labels)
plt.title(f"Prim's MST (weight = {total_weight})")

plt.tight_layout()
plt.show()

print(f"MST edges: {mst}")
print(f"Total MST weight: {total_weight}")

## Example 7: Kruskal's MST Algorithm

Kruskal's algorithm builds the MST by sorting all edges by weight and greedily adding edges that don't create a cycle. Uses **Union-Find** data structure for cycle detection.

**Time complexity:** $O(E \log E)$ dominated by sorting

**Union-Find operations:**
- `find(x)`: Find the root of component containing x
- `union(x, y)`: Merge components containing x and y

With path compression and union by rank, each operation is nearly $O(1)$.

In [ ]:
class UnionFind:
    """Union-Find with path compression and union by rank."""
    
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n
    
    def find(self, x: int) -> int:
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # Path compression
        return self.parent[x]
    
    def union(self, x: int, y: int) -> bool:
        """Union x and y. Returns True if they were in different sets."""
        px, py = self.find(x), self.find(y)
        if px == py:
            return False
        
        # Union by rank
        if self.rank[px] < self.rank[py]:
            px, py = py, px
        self.parent[py] = px
        if self.rank[px] == self.rank[py]:
            self.rank[px] += 1
        
        return True


def kruskal(n: int, edges: List[Tuple[int, int, float]]) -> List[Tuple[int, int, float]]:
    """
    Kruskal's algorithm.
    
    Args:
        n: Number of vertices
        edges: List of (u, v, weight)
        
    Returns:
        List of edges in MST
    """
    # Sort edges by weight
    sorted_edges = sorted(edges, key=lambda x: x[2])
    
    uf = UnionFind(n)
    mst = []
    
    for u, v, w in sorted_edges:
        if uf.union(u, v):  # If u and v in different components
            mst.append((u, v, w))
            if len(mst) == n - 1:
                break
    
    return mst


# Run Kruskal's on same graph
mst_kruskal = kruskal(6, edges_list)
total_weight_kruskal = sum(w for _, _, w in mst_kruskal)

print("Kruskal's MST:")
for u, v, w in mst_kruskal:
    print(f"  ({u}, {v}) weight={w}")
print(f"Total MST weight: {total_weight_kruskal}")
print(f"\nSame as Prim's: {total_weight_kruskal == total_weight}")

## Example 8: Topological Sort

Topological sort produces a linear ordering of vertices in a **DAG** (Directed Acyclic Graph) such that for every edge $(u, v)$, vertex $u$ comes before $v$.

**Two approaches:**
1. **Kahn's algorithm (BFS):** Process vertices with in-degree 0
2. **DFS-based:** Return reverse post-order

**Applications in ML:**
- Neural network forward pass ordering
- Computation graph execution
- Dependency resolution in build systems

In [ ]:
def topological_sort_kahn(adj_list: Dict[int, List[int]], 
                          vertices: List[int]) -> List[int]:
    """
    Kahn's algorithm (BFS-based).
    Returns empty list if graph has cycle.
    """
    # Compute in-degrees
    in_degree = {v: 0 for v in vertices}
    for u in adj_list:
        for v in adj_list[u]:
            in_degree[v] += 1
    
    # Start with vertices having in-degree 0
    queue = deque([v for v in vertices if in_degree[v] == 0])
    result = []
    
    while queue:
        u = queue.popleft()
        result.append(u)
        
        for v in adj_list.get(u, []):
            in_degree[v] -= 1
            if in_degree[v] == 0:
                queue.append(v)
    
    if len(result) != len(vertices):
        return []  # Cycle detected
    
    return result


def topological_sort_dfs(adj_list: Dict[int, List[int]], 
                         vertices: List[int]) -> List[int]:
    """DFS-based topological sort. Returns reverse post-order."""
    visited = set()
    result = []
    has_cycle = [False]
    in_stack = set()
    
    def dfs(u):
        if has_cycle[0]:
            return
        
        visited.add(u)
        in_stack.add(u)
        
        for v in adj_list.get(u, []):
            if v in in_stack:
                has_cycle[0] = True
                return
            if v not in visited:
                dfs(v)
        
        in_stack.remove(u)
        result.append(u)
    
    for v in vertices:
        if v not in visited:
            dfs(v)
    
    return [] if has_cycle[0] else result[::-1]


# Neural network computation graph
# 0: input, 1: conv1, 2: relu1, 3: conv2, 4: relu2, 5: fc, 6: output
nn_adj = {
    0: [1],      # input → conv1
    1: [2],      # conv1 → relu1
    2: [3],      # relu1 → conv2
    3: [4],      # conv2 → relu2
    4: [5],      # relu2 → fc
    5: [6],      # fc → output
    6: []
}
layer_names = ['input', 'conv1', 'relu1', 'conv2', 'relu2', 'fc', 'output']
vertices_nn = list(range(7))

order_kahn = topological_sort_kahn(nn_adj, vertices_nn)
order_dfs = topological_sort_dfs(nn_adj, vertices_nn)

print("Neural network computation graph:")
print(f"  Topological order (Kahn): {[layer_names[v] for v in order_kahn]}")
print(f"  Topological order (DFS):  {[layer_names[v] for v in order_dfs]}")

# Visualize
G_nn = nx.DiGraph(nn_adj)
plt.figure(figsize=(12, 3))
pos = {i: (i, 0) for i in range(7)}
nx.draw(G_nn, pos, labels={i: layer_names[i] for i in range(7)}, 
        node_color='lightskyblue', node_size=1500, font_size=10, 
        font_weight='bold', arrows=True, arrowsize=20)
plt.title("Neural Network Computation Graph")
plt.tight_layout()
plt.show()

## Example 9: Strongly Connected Components (Kosaraju's Algorithm)

A **Strongly Connected Component (SCC)** is a maximal set of vertices where every vertex is reachable from every other vertex.

**Kosaraju's Algorithm:**
1. Perform DFS on original graph, record finish order
2. Create transpose graph (reverse all edges)
3. Perform DFS on transpose in reverse finish order

**Time complexity:** $O(V + E)$

**Applications:** Analyzing strongly connected web pages, social network analysis, 2-SAT problem

In [ ]:
def kosaraju_scc(adj_list: Dict[int, List[int]], 
                 vertices: List[int]) -> List[List[int]]:
    """
    Kosaraju's algorithm for SCCs.
    
    Returns:
        List of SCCs (each SCC is a list of vertices)
    """
    # Step 1: DFS to get finish order
    visited = set()
    finish_order = []
    
    def dfs1(u):
        visited.add(u)
        for v in adj_list.get(u, []):
            if v not in visited:
                dfs1(v)
        finish_order.append(u)
    
    for v in vertices:
        if v not in visited:
            dfs1(v)
    
    # Step 2: Create transpose graph
    transpose = defaultdict(list)
    for u in adj_list:
        for v in adj_list[u]:
            transpose[v].append(u)
    
    # Step 3: DFS on transpose in reverse finish order
    visited = set()
    sccs = []
    
    def dfs2(u, component):
        visited.add(u)
        component.append(u)
        for v in transpose.get(u, []):
            if v not in visited:
                dfs2(v, component)
    
    for v in reversed(finish_order):
        if v not in visited:
            component = []
            dfs2(v, component)
            sccs.append(component)
    
    return sccs


# Graph with 3 SCCs
scc_adj = {
    0: [1],
    1: [2],
    2: [0, 3],  # {0, 1, 2} form an SCC
    3: [4],
    4: [5],
    5: [3],     # {3, 4, 5} form an SCC
    6: [5, 7],
    7: [6]      # {6, 7} form an SCC
}

sccs = kosaraju_scc(scc_adj, list(range(8)))

print("Strongly Connected Components:")
for i, scc in enumerate(sccs):
    print(f"  SCC {i + 1}: {sorted(scc)}")

# Visualize
G_scc = nx.DiGraph(scc_adj)
plt.figure(figsize=(10, 6))
pos = nx.spring_layout(G_scc, seed=42)

# Color nodes by SCC
colors = ['lightcoral', 'lightgreen', 'lightskyblue', 'lightyellow']
node_colors = []
for node in G_scc.nodes():
    for i, scc in enumerate(sccs):
        if node in scc:
            node_colors.append(colors[i % len(colors)])
            break

nx.draw(G_scc, pos, with_labels=True, node_color=node_colors, 
        node_size=500, font_size=14, font_weight='bold', arrows=True)
plt.title("Graph with SCCs (colored by component)")
plt.show()

## Example 10: Graph Coloring (Greedy)

Graph coloring assigns colors to vertices such that **no two adjacent vertices share the same color**. The greedy algorithm finds a valid coloring (not necessarily optimal).

**Chromatic number** $\chi(G)$: Minimum number of colors needed.

**Greedy bound:** Uses at most $\Delta + 1$ colors, where $\Delta$ is the maximum degree.

**Applications:** Register allocation in compilers, scheduling, map coloring, frequency assignment

In [ ]:
def greedy_coloring(adj_list: Dict[int, List[int]], 
                    vertices: List[int]) -> Dict[int, int]:
    """
    Greedy graph coloring.
    Assigns smallest available color to each vertex.
    """
    colors = {}
    
    for v in vertices:
        # Find colors used by neighbors
        neighbor_colors = set()
        for u in adj_list.get(v, []):
            if u in colors:
                neighbor_colors.add(colors[u])
        
        # Find smallest available color
        color = 0
        while color in neighbor_colors:
            color += 1
        
        colors[v] = color
    
    return colors


def is_valid_coloring(adj_list: Dict[int, List[int]], colors: Dict[int, int]) -> bool:
    """Check if coloring is valid."""
    for u in adj_list:
        for v in adj_list[u]:
            if colors.get(u) == colors.get(v):
                return False
    return True


# Petersen-like graph
color_adj = {
    0: [1, 4, 5],
    1: [0, 2, 6],
    2: [1, 3, 7],
    3: [2, 4, 8],
    4: [3, 0, 9],
    5: [0, 7, 8],
    6: [1, 8, 9],
    7: [2, 5, 9],
    8: [3, 5, 6],
    9: [4, 6, 7]
}
vertices_color = list(range(10))

colors = greedy_coloring(color_adj, vertices_color)
num_colors = len(set(colors.values()))
max_degree = max(len(color_adj.get(v, [])) for v in vertices_color)

print(f"Coloring result:")
print(f"  {colors}")
print(f"  Colors used: {num_colors}")
print(f"  Max degree Δ: {max_degree}")
print(f"  Valid coloring: {is_valid_coloring(color_adj, colors)}")

# Visualize
G_color = nx.Graph(color_adj)
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G_color, seed=42)

color_map = plt.cm.Set3
node_colors = [color_map(colors[v] / num_colors) for v in G_color.nodes()]

nx.draw(G_color, pos, with_labels=True, node_color=node_colors, 
        node_size=600, font_size=14, font_weight='bold')
plt.title(f"Greedy Graph Coloring ({num_colors} colors)")
plt.show()

## Example 11: Maximum Flow (Ford-Fulkerson with Edmonds-Karp)

The **Maximum Flow Problem** finds the maximum amount of flow that can be sent from a source to a sink in a network with edge capacities.

**Ford-Fulkerson Method:**
1. While there exists an augmenting path from source to sink:
   - Find the path (BFS for Edmonds-Karp)
   - Compute bottleneck capacity
   - Update residual graph

**Time complexity:** $O(VE^2)$ for Edmonds-Karp

**Applications:** Bipartite matching, network routing, image segmentation

In [ ]:
def ford_fulkerson(capacity: Dict[Tuple[int, int], float],
                   source: int,
                   sink: int,
                   vertices: List[int]) -> Tuple[float, Dict]:
    """
    Ford-Fulkerson algorithm with BFS (Edmonds-Karp).
    
    Args:
        capacity: {(u, v): capacity} edge capacities
        source: Source vertex
        sink: Sink vertex
        vertices: All vertices
        
    Returns:
        max_flow: Maximum flow value
        flow: Flow on each edge
    """
    # Build adjacency list for residual graph
    adj = defaultdict(list)
    for (u, v) in capacity:
        adj[u].append(v)
        adj[v].append(u)  # Reverse edge
    
    # Residual capacity
    residual = defaultdict(float)
    for (u, v), cap in capacity.items():
        residual[(u, v)] = cap
    
    def bfs_path():
        """Find augmenting path using BFS."""
        visited = {source}
        parent = {source: None}
        queue = deque([source])
        
        while queue:
            u = queue.popleft()
            for v in adj[u]:
                if v not in visited and residual[(u, v)] > 0:
                    visited.add(v)
                    parent[v] = u
                    if v == sink:
                        # Reconstruct path
                        path = []
                        curr = sink
                        while parent[curr] is not None:
                            path.append((parent[curr], curr))
                            curr = parent[curr]
                        return path[::-1]
                    queue.append(v)
        return None
    
    max_flow = 0
    while True:
        path = bfs_path()
        if path is None:
            break
        
        # Find bottleneck
        bottleneck = min(residual[(u, v)] for u, v in path)
        
        # Update residual graph
        for u, v in path:
            residual[(u, v)] -= bottleneck
            residual[(v, u)] += bottleneck
        
        max_flow += bottleneck
    
    # Compute actual flow
    flow = {}
    for (u, v), cap in capacity.items():
        flow[(u, v)] = cap - residual[(u, v)]
    
    return max_flow, flow


# Flow network
capacity = {
    (0, 1): 10, (0, 2): 10,
    (1, 2): 2, (1, 3): 4, (1, 4): 8,
    (2, 4): 9,
    (3, 5): 10,
    (4, 3): 6, (4, 5): 10
}

max_flow, flow = ford_fulkerson(capacity, source=0, sink=5, vertices=list(range(6)))

print(f"Maximum flow: {max_flow}")
print(f"\nFlow on each edge:")
for (u, v), f in sorted(flow.items()):
    if f > 0:
        print(f"  {u} → {v}: {f}/{capacity[(u, v)]}")

# Visualize
G_flow = nx.DiGraph()
for (u, v), cap in capacity.items():
    G_flow.add_edge(u, v, capacity=cap, flow=flow[(u, v)])

plt.figure(figsize=(10, 6))
pos = {0: (0, 1), 1: (1, 2), 2: (1, 0), 3: (2, 2), 4: (2, 0), 5: (3, 1)}
nx.draw(G_flow, pos, with_labels=True, node_color='lightblue', 
        node_size=600, font_size=14, font_weight='bold', arrows=True)
edge_labels = {(u, v): f"{flow[(u, v)]}/{capacity[(u, v)]}" 
               for (u, v) in capacity}
nx.draw_networkx_edge_labels(G_flow, pos, edge_labels=edge_labels, font_size=10)
plt.title(f"Maximum Flow Network (max flow = {max_flow})")
plt.show()

## Example 12: PageRank Algorithm

PageRank computes the importance of each node based on the link structure of the graph. Originally developed for web page ranking.

**Formula:**
$$PR(v) = \frac{1 - d}{n} + d \sum_{u \in In(v)} \frac{PR(u)}{|Out(u)|}$$

Where:
- $d$ is the damping factor (typically 0.85)
- $n$ is the number of vertices
- $In(v)$ are vertices pointing to $v$
- $Out(u)$ are vertices $u$ points to

**Applications:** Web page ranking, social network analysis, node importance in GNNs

In [ ]:
def pagerank(adj_list: Dict[int, List[int]], 
             n: int,
             damping: float = 0.85,
             max_iter: int = 100,
             tol: float = 1e-6) -> np.ndarray:
    """
    Compute PageRank using power iteration.
    
    Args:
        adj_list: {vertex: [outgoing neighbors]}
        n: Number of vertices
        damping: Damping factor (typically 0.85)
        max_iter: Maximum iterations
        tol: Convergence tolerance
        
    Returns:
        PageRank scores for each vertex
    """
    # Initialize
    pr = np.ones(n) / n
    
    # Compute out-degrees
    out_degree = np.array([len(adj_list.get(u, [])) for u in range(n)])
    
    # Handle dangling nodes (no outgoing edges)
    dangling = out_degree == 0
    
    # Build incoming edges
    incoming = defaultdict(list)
    for u in adj_list:
        for v in adj_list[u]:
            incoming[v].append(u)
    
    # Power iteration
    for iteration in range(max_iter):
        pr_new = np.ones(n) * (1 - damping) / n
        
        # Dangling node contribution
        dangling_sum = pr[dangling].sum()
        pr_new += damping * dangling_sum / n
        
        # Regular contribution
        for v in range(n):
            for u in incoming[v]:
                pr_new[v] += damping * pr[u] / out_degree[u]
        
        # Check convergence
        diff = np.abs(pr_new - pr).sum()
        pr = pr_new
        
        if diff < tol:
            print(f"Converged in {iteration + 1} iterations")
            break
    
    return pr


# Web graph example
web_adj = {
    0: [1, 2],    # Page 0 links to 1, 2
    1: [2],       # Page 1 links to 2
    2: [0],       # Page 2 links to 0
    3: [2]        # Page 3 links to 2
}

pr = pagerank(web_adj, 4)

print(f"\nPageRank scores:")
for v in range(4):
    print(f"  Page {v}: {pr[v]:.4f}")
print(f"\nMost important page: {np.argmax(pr)}")

# Compare with NetworkX
G_pr = nx.DiGraph(web_adj)
pr_nx = nx.pagerank(G_pr, alpha=0.85)
print(f"\nNetworkX PageRank: {pr_nx}")

# Visualize
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G_pr, seed=42)
node_sizes = [1000 + 3000 * pr[v] for v in G_pr.nodes()]
nx.draw(G_pr, pos, with_labels=True, node_color='lightyellow', 
        node_size=node_sizes, font_size=14, font_weight='bold', arrows=True)
plt.title("Web Graph (node size ∝ PageRank)")
plt.show()

## Summary

This notebook covered 12 fundamental graph algorithms:

| Algorithm | Time Complexity | Key Application |
|-----------|-----------------|------------------|
| BFS | $O(V + E)$ | Shortest path (unweighted), GNN neighborhoods |
| DFS | $O(V + E)$ | Topological sort, cycle detection |
| Dijkstra | $O((V + E) \log V)$ | Shortest path (non-negative weights) |
| Bellman-Ford | $O(VE)$ | Negative weights, negative cycle detection |
| Floyd-Warshall | $O(V^3)$ | All-pairs shortest paths |
| Prim | $O(E \log V)$ | Minimum spanning tree |
| Kruskal | $O(E \log E)$ | Minimum spanning tree |
| Topological Sort | $O(V + E)$ | DAG ordering, neural network execution |
| Kosaraju SCC | $O(V + E)$ | Strongly connected components |
| Graph Coloring | $O(V + E)$ | Scheduling, register allocation |
| Ford-Fulkerson | $O(VE^2)$ | Maximum flow, bipartite matching |
| PageRank | $O(V + E)$ per iter | Node importance, web ranking |

These algorithms form the foundation for more advanced graph-based machine learning methods like Graph Neural Networks (GNNs), knowledge graphs, and recommendation systems.